In [1]:
!pip install catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import HistGradientBoostingClassifier
from catboost import CatBoostClassifier


In [3]:
# Load dataset
data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset shape:", X.shape)
print("Number of classes:", len(np.unique(y)))


Dataset shape: (569, 30)
Number of classes: 2


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (455, 30)
Test size: (114, 30)


In [5]:
hpboost = HistGradientBoostingClassifier(
    max_iter=200,
    random_state=42
)

start = time.time()
hpboost.fit(X_train, y_train)
hp_time = time.time() - start

y_hp = hpboost.predict(X_test)

hp_acc = accuracy_score(y_test, y_hp)

print("HPBoost Accuracy:", hp_acc)
print("Training Time:", hp_time)


HPBoost Accuracy: 0.9736842105263158
Training Time: 0.6419312953948975


In [6]:
catboost = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    verbose=0,
    random_state=42
)

start = time.time()
catboost.fit(X_train, y_train)
cat_time = time.time() - start

y_cat = catboost.predict(X_test)

cat_acc = accuracy_score(y_test, y_cat)

print("CatBoost Accuracy:", cat_acc)
print("Training Time:", cat_time)


CatBoost Accuracy: 0.9736842105263158
Training Time: 3.9610400199890137


In [7]:
print("=== HPBoost Report ===")
print(classification_report(y_test, y_hp))

print("\n=== CatBoost Report ===")
print(classification_report(y_test, y_cat))


=== HPBoost Report ===
              precision    recall  f1-score   support

           0       1.00      0.93      0.96        42
           1       0.96      1.00      0.98        72

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114


=== CatBoost Report ===
              precision    recall  f1-score   support

           0       1.00      0.93      0.96        42
           1       0.96      1.00      0.98        72

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114



In [8]:
results = pd.DataFrame({
    "Model": ["HPBoost (HistGradientBoosting)", "CatBoost"],
    "Accuracy": [hp_acc, cat_acc],
    "Training Time (sec)": [hp_time, cat_time]
})

results


,Model,Accuracy,Training Time (sec)
0,HPBoost (HistGradientBoosting),0.973684,0.641931
1,CatBoost,0.973684,3.961040


Model Comparison Summary:

- **HPBoost** achieved higher accuracy.
- **HPBoost** trained faster.

Interpretation:
- **CatBoost** is strong for stability and complex patterns.
- **HPBoost** focuses on faster optimized training.